In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("feature_eng_df_final.csv")

In [4]:
df.head()

,age,education,monthly_salary,years_of_employment,company_type,house_type,monthly_rent,family_size,dependents,school_fees,...,affordability_ratio,credit_score_category,credit_score_numeric,combined_credit_risk,employment_tenure_category,is_long_term_employed,income_per_family_member,savings_to_income_ratio,credit_stability_score,loan_affordability_index
0,38.0,3.0,11.321777,0.641854,2.0,0.0,1.280128,3,2,0.0,...,-2.443993,Fair,2,3,Entry-level,0,3.773926,0.042580,423.623565,1.205905
1,38.0,1.0,9.975855,2.079442,4.0,1.0,-0.814566,2,1,5100.0,...,-513.331366,Good,1,2,Mid-level,1,4.987927,-0.177067,1484.721261,1.178826
2,38.0,3.0,11.363276,1.916923,0.0,2.0,-0.814566,4,3,0.0,...,-1.321418,Fair,2,2,Mid-level,1,2.840819,0.346565,1245.999698,1.111593
3,58.0,0.0,11.109473,1.163151,2.0,2.0,-0.814566,5,4,11400.0,...,-1027.461671,Good,1,1,Entry-level,0,2.221895,0.200884,796.758305,1.136398
4,48.0,3.0,10.956073,1.481605,2.0,1.0,-0.814566,4,3,9400.0,...,-859.501890,Excellent,0,0,Mid-level,0,2.739018,-0.153537,1140.835497,1.135187


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

In [6]:
# dropping credit score category since it already encoded as new column credit_score_numeric
df.drop('credit_score_category',axis=1,inplace=True)

In [7]:
#let's encode the binned age group column derived from age column in to ordinal encoder
df['age_group'].unique()
age_group_order = ['25-34','35-44', '45-54',, '55-64']
from sklearn.preprocessing import OrdinalEncoder
age_group_encoder = OrdinalEncoder(categories=[age_group_order])
df['age_group'] = age_group_encoder.fit_transform(df[['age_group']])
print("after encoding df['age_group']",df['age_group'].unique())
df.drop('age',axis=1,inplace=True)

after encoding df['age_group'] [1. 2. 3. 0.]


In [8]:
#let encode binned employment experience type from year_of_employment column 
df['employment_tenure_category'].unique()
employment_order = ['Entry-level', 'Mid-level', 'Experienced']
emp_category_encoder = OrdinalEncoder(categories=[employment_order])
df['employment_tenure_category'] = emp_category_encoder.fit_transform(df[['employment_tenure_category']])
print("after encoding emp tenure category ",df['employment_tenure_category'].unique())
df.drop('years_of_employment',axis=1,inplace=True)

after encoding emp tenure category  [0. 1. 2.]


In [9]:
X = df.drop(['max_monthly_emi','emi_eligibility'],axis=1)
y= df['emi_eligibility']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("x train shape",X_train.shape)
print("X test shape",X_test.shape)
print("y train shape",y_train.shape)
print("y test shape",y_test.shape)

x train shape (322243, 40)
X test shape (80561, 40)
y train shape (322243,)
y test shape (80561,)


In [10]:
rf_model = RandomForestClassifier(n_estimators=200,random_state=42)
rf_model.fit(X_train,y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
rf_y_pred = rf_model.predict(X_test)

In [12]:
# Get the Analysis or Metrics
rf_accuracy = accuracy_score(y_test,rf_y_pred)
print("accuracy--",rf_accuracy)
rf_precision = precision_score(y_test,rf_y_pred,average='weighted')
print("precision--",rf_precision)
rf_recall = recall_score(y_test,rf_y_pred,average="weighted")
print("recall---",rf_recall)
rf_f1_score = f1_score(y_test,rf_y_pred,average="weighted")
print("f1 score --",rf_f1_score)
rf_y_pred_proba = rf_model.predict_proba(X_test)
roc_auc_scr    = roc_auc_score(y_test,rf_y_pred_proba,multi_class='ovo',average="weighted")
print("roc_auc_scr",roc_auc_scr) 

accuracy-- 0.9321135537046462
precision-- 0.9236157981091521
recall--- 0.9321135537046462
f1 score -- 0.9131994130398817
roc_auc_scr 0.9620784782573534


In [13]:
from sklearn.metrics import confusion_matrix
confusion_mat = confusion_matrix(y_test,rf_y_pred)
print(confusion_matrix(y_test,rf_y_pred))

[[12948    25  1515]
 [ 1040    86  2318]
 [  565     6 62058]]


In [14]:
rf_train_pred = rf_model.predict(X_train)

In [16]:
rf_train_accuracy = accuracy_score(y_train,rf_train_pred)
print("accuracy--",rf_train_accuracy)
rf_t_precision = precision_score(y_train,rf_train_pred,average='weighted')
print("precision--",rf_t_precision)
rf_t_recall = recall_score(y_train,rf_train_pred,average="weighted")
print("recall---",rf_t_recall)
rf_t_f1_score = f1_score(y_train,rf_train_pred,average="weighted")
print("f1 score --",rf_t_f1_score)


accuracy-- 1.0
precision-- 1.0
recall--- 1.0
f1 score -- 1.0


In [38]:
conf_mat_train = confusion_matrix(y_train,rf_train_pred)
print(conf_mat_train)

[[ 59594      0      0]
 [     0  13952      0]
 [     0      0 248697]]


In [ ]:
rf_yt_pred_proba = rf_model.predict_proba(X_train)
roc_auc_scr    = roc_auc_score(y_test,rf_y_pred_proba,multi_class='ovo',average="weighted")
print("roc_auc_scr",roc_auc_scr) 

In [20]:
# Let's Do Hyper parameter tuning 
# important parameters for 
# 1. n_estimators, 2. max_depth, 3. min_sample_split, 4. min_sample_leaf, 5. max_features
# which one tuning startegy to choose - GridSearchCV or RandomizedSearchCV
          #  large dataset , many parameters,limited Computations - RandomizedSearchCV 
          #  small dataset , few parameter - GridSearchCV
          # 300K dataset = RandomizedSearchCv would be best
        
#prepare params_grid for hyper tuning 
params_dist = {
    "n_estimators":[200,300,500]
    ,"max_depth":[None,10,20,30]
    ,"min_samples_split":[2,5,10]
}
rfc = RandomForestClassifier(random_state=42,min_samples_leaf=1,max_features="sqrt")
random_search = RandomizedSearchCV(rfc,param_distributions=params_dist,n_iter=5,cv=5,n_jobs=3,random_state=42)


In [21]:
random_search.fit(X_train,y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [None, 10, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [200, 300, ...]}"
,n_iter,5
,scoring,None
,n_jobs,3
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [29]:
print(random_search.best_params_)

{'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 30}


In [22]:
best_rf_model = random_search.best_estimator_

In [24]:
y_rfb_prdict = best_rf_model.predict(X_test)

In [26]:
rfb_accuracy = accuracy_score(y_test,y_rfb_prdict)
print("best accuracy",rfb_accuracy)
rfb_precision = precision_score(y_test,y_rfb_prdict,average="weighted")
print("best precision",rfb_precision)
rfb_recall = recall_score(y_test,y_rfb_prdict,average="weighted")
print("best recall",rfb_recall)
rfb_f1_score = f1_score(y_test,y_rfb_prdict,average="weighted")
print("best f1 score",rfb_f1_score)

best accuracy 0.9317535780340364
best precision 0.9241603794199482
best recall 0.9317535780340364
best f1 score 0.9124927130491087


In [27]:
confusion_mat_rfb = confusion_matrix(y_test,y_rfb_prdict)
print("confusion matrix")
print(confusion_mat_rfb)

confusion matrix
[[12922    19  1547]
 [ 1058    69  2317]
 [  554     3 62072]]


In [28]:
y_rfb_predict_prob = best_rf_model.predict_proba(X_test)
roc_auc_rfb = roc_auc_score(y_test,y_rfb_predict_prob,multi_class="ovo",average="weighted")
print("roc auc score--",roc_auc_rfb)

roc auc score-- 0.9625761916023343


In [33]:
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Random Forest Classifier experiment")
with mlflow.start_run():
    mlflow.log_params(random_search.best_params_)
    mlflow.log_metric("best_cv_score",random_search.best_score_)
    mlflow.log_metric("accuracy",rfb_accuracy)
    mlflow.log_metric("precision",rfb_precision)
    mlflow.log_metric("recall",rfb_recall)
    mlflow.log_metric("f1_score",rfb_f1_score)
    mlflow.log_metric("roc_auc_score",roc_auc_rfb)
    mlflow.sklearn.log_model(random_search.best_estimator_,"best_model")
    print("completed logging in mlflow")

2026/02/15 17:18:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
C:\Users\RAM\miniconda3\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


completed logging in mlflow
🏃 View run puzzled-grouse-709 at: http://127.0.0.1:5000/#/experiments/3/runs/57e914c135c246f497a48a2b044b6a1c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
